# Tutorial 11: Cross-Species Ortholog Embeddings

Cross-species comparisons use one `BioEmbedder` per organism and the same
`BioEmbedder.embed(...)` call for each species.


In [ ]:
from embpy import BioEmbedder
import anndata as ad
import numpy as np
import pandas as pd

RUN_EMBEDDING = False
human = BioEmbedder(device="auto", organism="human")
mouse = BioEmbedder(device="auto", organism="mouse")
zebrafish = BioEmbedder(device="auto", organism="zebrafish")


## 1. Embed ortholog panels species by species


In [ ]:
orthologs = {
    "human": ["TP53", "BRCA1", "EGFR"],
    "mouse": ["Trp53", "Brca1", "Egfr"],
    "zebrafish": ["tp53", "brca1", "egfra"],
}

if RUN_EMBEDDING:
    human_adata = human.embed(
        orthologs["human"], entity_type="gene", id_type="symbol",
        model="esm2_650M", output="anndata", key="X_esm2_650M",
    )
    mouse_adata = mouse.embed(
        orthologs["mouse"], entity_type="gene", id_type="symbol",
        model="esm2_650M", output="anndata", key="X_esm2_650M",
    )
    fish_adata = zebrafish.embed(
        orthologs["zebrafish"], entity_type="gene", id_type="symbol",
        model="esm2_650M", output="anndata", key="X_esm2_650M",
    )
    print(human_adata.varm["X_esm2_650M"].shape)


## 2. Combine aligned outputs for visualization


In [ ]:
if RUN_EMBEDDING:
    matrices = []
    rows = []
    for species, species_adata in [
        ("human", human_adata),
        ("mouse", mouse_adata),
        ("zebrafish", fish_adata),
    ]:
        X = species_adata.varm["X_esm2_650M"]
        matrices.append(X)
        rows.extend({"species": species, "canonical_id": x} for x in species_adata.var_names)

    cross = ad.AnnData(
        X=np.vstack(matrices),
        obs=pd.DataFrame(rows),
    )
    cross.obsm["X_esm2_650M"] = cross.X.copy()
    print(cross)


## 3. The same pattern works for DNA models


In [ ]:
if RUN_EMBEDDING:
    human_dna = human.embed(
        orthologs["human"],
        entity_type="gene",
        id_type="symbol",
        model="nt_v2_100m",
        output="anndata",
        key="X_nt_v2_100m",
    )
    print(human_dna.varm["X_nt_v2_100m"].shape)
